In [1]:
# ============================================================
# 1. 라이브러리 설치
# ============================================================

!pip install -q "plotly>=6.0" geopandas pyogrio ipywidgets

import pandas as pd
import numpy as np
import geopandas as gpd
import plotly.express as px
import json
import zipfile

from pathlib import Path
from google.colab import files
from IPython.display import display

from google.colab import output
output.enable_custom_widget_manager()

In [2]:
# ============================================================
# 2. 사업자 데이터 ZIP 업로드
# ============================================================

uploaded = files.upload()

DATA_ZIP = next(
    name for name in uploaded.keys()
    if name.lower().endswith(".zip")
)

print("업로드된 파일:", DATA_ZIP)

Saving 부산_사업자현황_통합_202201_202606_업종6분류.zip to 부산_사업자현황_통합_202201_202606_업종6분류 (3).zip
업로드된 파일: 부산_사업자현황_통합_202201_202606_업종6분류 (3).zip


In [3]:
# ============================================================
# 3. ZIP 내부 CSV 읽기
# ============================================================

with zipfile.ZipFile(DATA_ZIP, "r") as z:
    csv_files = [
        name for name in z.namelist()
        if name.lower().endswith(".csv")
    ]

    print("CSV:", csv_files)

    with z.open(csv_files[0]) as f:
        df = pd.read_csv(f, encoding="utf-8-sig")

print(df.shape)
display(df.head())

CSV: ['부산_사업자현황_통합_202201_202606_업종6분류.csv']
(91264, 11)


,연월,구분3,업종대분류,구분2,구분1,항목1,당월①,전월②,증감율(①/②),전년동월③,증감율(①/③)
0,202201,업종전체,NaN,부산광역시,부산광역시 합계,NaN,"178,966","177,516",100.82,"167,924",106.58
1,202201,업종전체,NaN,부산광역시,중구,NaN,"6,367","6,331",100.57,"6,354",100.20
2,202201,업종전체,NaN,부산광역시,서구,NaN,"4,715","4,690",100.53,"4,606",102.37
3,202201,업종전체,NaN,부산광역시,동구,NaN,"6,310","6,272",100.61,"6,072",103.92
4,202201,업종전체,NaN,부산광역시,영도구,NaN,"4,453","4,430",100.52,"4,246",104.88


In [4]:
# ============================================================
# 4. 데이터 전처리
# ============================================================

raw = df.copy()

# ----------------------------
# 숫자형 변환
# ----------------------------

num_cols = [
    "당월①",
    "전월②",
    "전년동월③"
]

for col in num_cols:
    raw[col] = (
        raw[col]
        .astype(str)
        .str.replace(",", "", regex=False)
    )

    raw[col] = pd.to_numeric(
        raw[col],
        errors="coerce"
    )


# ----------------------------
# 날짜 변수
# ----------------------------

raw["연월"] = raw["연월"].astype(str)

raw["기준일"] = pd.to_datetime(
    raw["연월"],
    format="%Y%m"
)

raw["연도"] = raw["기준일"].dt.year
raw["월"] = raw["기준일"].dt.month
raw["연월표시"] = raw["기준일"].dt.strftime("%Y-%m")


# 부산 전체 합계는 지역 지도에서는 제외
raw = raw[
    raw["구분1"] != "부산광역시 합계"
].copy()


print(
    raw["기준일"].min(),
    "~",
    raw["기준일"].max()
)

print(raw["구분1"].unique())

2022-01-01 00:00:00 ~ 2026-06-01 00:00:00
['중구' '서구' '동구' '영도구' '부산진구' '동래구' '남구' '북구' '해운대구' '사하구' '금정구' '강서구'
 '연제구' '수영구' '사상구' '기장군']


In [5]:
# ============================================================
# 5. 전체 업종 데이터
# ============================================================

overall = (
    raw[
        raw["구분3"] == "업종전체"
    ][[
        "연월",
        "기준일",
        "연도",
        "월",
        "연월표시",
        "구분1",
        "당월①",
        "전월②",
        "전년동월③"
    ]]
    .rename(columns={
        "구분1": "지역",
        "당월①": "사업자수",
        "전월②": "전월사업자수",
        "전년동월③": "전년동월사업자수"
    })
)

overall["업종"] = "전체"

In [6]:
# ============================================================
# 6. 6개 업종대분류별 집계
# ============================================================

major = (
    raw[
        raw["업종대분류"].notna()
    ]
    .groupby([
        "연월",
        "기준일",
        "연도",
        "월",
        "연월표시",
        "구분1",
        "업종대분류"
    ], as_index=False)
    .agg(
        사업자수=("당월①", "sum"),
        전월사업자수=("전월②", "sum"),
        전년동월사업자수=("전년동월③", "sum")
    )
    .rename(columns={
        "구분1": "지역",
        "업종대분류": "업종"
    })
)

In [7]:
# ============================================================
# 7. 전체 + 업종대분류 통합
# ============================================================

viz = pd.concat(
    [overall, major],
    ignore_index=True
)


# 전월 대비 성장률
viz["전월성장률"] = (
    viz["사업자수"]
    / viz["전월사업자수"]
    - 1
) * 100


# 전년동월 대비 성장률
viz["전년동월성장률"] = (
    viz["사업자수"]
    / viz["전년동월사업자수"]
    - 1
) * 100


# 2022-01 기준 성장지수
viz = viz.sort_values([
    "지역",
    "업종",
    "기준일"
])

viz["기준월사업자수"] = (
    viz
    .groupby(["지역", "업종"])
    ["사업자수"]
    .transform("first")
)


viz["성장지수"] = (
    viz["사업자수"]
    / viz["기준월사업자수"]
    * 100
)


viz["누적성장률"] = (
    viz["성장지수"] - 100
)


display(viz.head())

,연월,기준일,연도,월,연월표시,지역,사업자수,전월사업자수,전년동월사업자수,업종,전월성장률,전년동월성장률,기준월사업자수,성장지수,누적성장률
864,202201,2022-01-01,2022,1,2022-01,강서구,1802,1775,1573,서비스업,1.521127,14.558169,1802,100.000000,0.000000
960,202202,2022-02-01,2022,2,2022-02,강서구,1832,1802,1591,서비스업,1.664817,15.147706,1802,101.664817,1.664817
1056,202203,2022-03-01,2022,3,2022-03,강서구,1859,1832,1617,서비스업,1.473799,14.965986,1802,103.163152,3.163152
1152,202204,2022-04-01,2022,4,2022-04,강서구,1875,1859,1636,서비스업,0.860678,14.608802,1802,104.051054,4.051054
1248,202205,2022-05-01,2022,5,2022-05,강서구,1892,1875,1637,서비스업,0.906667,15.577276,1802,104.994451,4.994451


| 지표        | 의미                       |
| --------- | ------------------------ |
| `사업자수`    | 해당 월 실제 사업자 규모           |
| `전월성장률`   | 바로 전월 대비 성장              |
| `전년동월성장률` | 계절효과를 줄인 YoY 성장          |
| `누적성장률`   | 2022년 1월 대비 지금 얼마나 성장했는지 |
| `성장지수`    | 2022년 1월 = 100           |


In [8]:
# ============================================================
# 8. 부산 행정구역 SHP 사
# ============================================================

shp_files = list(
    Path("/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/부산 시군구").rglob("*.shp")
)

print(shp_files)

[PosixPath('/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/부산 시군구/BND_SIGUNGU_PG.shp')]


In [9]:
SHP_PATH = shp_files[0]

gu = gpd.read_file(SHP_PATH)

print(gu.crs)
print(gu.columns)

display(gu.head())

EPSG:5186
Index(['BASE_DATE', 'SIGUNGU_CD', 'SIGUNGU_NM', 'geometry'], dtype='object')


,BASE_DATE,SIGUNGU_CD,SIGUNGU_NM,geometry
0,20250630,11010,종로구,"POLYGON ((197800.72 559064.245, 197782.581 558..."
1,20250630,11020,중구,"POLYGON ((202043.919 552491.141, 202063.481 55..."
2,20250630,11030,용산구,"POLYGON ((197275.979 550595.299, 197275.977 55..."
3,20250630,11040,성동구,"POLYGON ((203535.345 552606.268, 203569.567 55..."
4,20250630,11050,광진구,"POLYGON ((208981.056 552544.566, 209031.744 55..."


In [10]:
# ============================================================
# 9. 행정구역 이름 정리
# ============================================================

gu = gu.to_crs(epsg=4326)


# 일반적으로 사용되는 행정구 컬럼명 자동 탐색
candidate_cols = [
    "SIGUNGU_NM",
    "SIG_KOR_NM",
    "SGG_NM",
    "ADM_NM",
    "구군명",
    "지역"
]

name_col = next(
    (
        col for col in candidate_cols
        if col in gu.columns
    ),
    None
)


if name_col is None:
    raise ValueError(
        f"행정구 이름 컬럼을 찾지 못했습니다.\n현재 컬럼: {gu.columns.tolist()}"
    )


gu["지역"] = (
    gu[name_col]
    .astype(str)
    .str.replace(
        "부산광역시",
        "",
        regex=False
    )
    .str.strip()
)


BUSAN_REGIONS = [
    "중구",
    "서구",
    "동구",
    "영도구",
    "부산진구",
    "동래구",
    "남구",
    "북구",
    "해운대구",
    "사하구",
    "금정구",
    "강서구",
    "연제구",
    "수영구",
    "사상구",
    "기장군"
]


gu = gu[
    gu["지역"].isin(BUSAN_REGIONS)
].copy()


# 동일 구역이 여러 Polygon이면 하나로 합침
gu = gu.dissolve(
    by="지역",
    as_index=False
)


print("지도 지역 수:", gu["지역"].nunique())

display(gu[["지역", "geometry"]])

지도 지역 수: 16


,지역,geometry
0,강서구,"MULTIPOLYGON (((128.78091 35.00886, 128.78093 ..."
1,금정구,"POLYGON ((129.10621 35.30646, 129.1064 35.3063..."
2,기장군,"MULTIPOLYGON (((129.21242 35.17907, 129.21242 ..."
3,남구,"MULTIPOLYGON (((129.10569 35.08442, 129.10552 ..."
4,동구,"MULTIPOLYGON (((126.93202 35.16064, 126.93203 ..."
5,동래구,"POLYGON ((129.07816 35.22579, 129.07821 35.225..."
6,부산진구,"POLYGON ((129.04028 35.1994, 129.04033 35.1993..."
7,북구,"MULTIPOLYGON (((126.91256 35.25829, 126.91279 ..."
8,사상구,"POLYGON ((128.99112 35.19384, 128.99121 35.193..."
9,사하구,"MULTIPOLYGON (((128.95178 34.88438, 128.95178 ..."


In [11]:
# GeoJSON 으로 변
busan_geojson = json.loads(
    gu.to_json()
)

In [12]:
# ============================================================
# 10. 연도별 대표 데이터
# ============================================================

annual = (
    viz
    .sort_values("기준일")
    .groupby(
        ["연도", "지역", "업종"],
        group_keys=False
    )
    .tail(1)
    .copy()
)

annual["연도표시"] = (
    annual["연도"].astype(str)
)

In [13]:
# ============================================================
# 11. 연도별 지도 함수
# ============================================================

def annual_growth_map(
    industry="전체",
    metric="전년동월성장률"
):

    data = annual[
        annual["업종"] == industry
    ].copy()


    limit = np.nanmax(
        np.abs(data[metric])
    )

    if pd.isna(limit) or limit == 0:
        limit = 1


    fig = px.choropleth_map(
        data,
        geojson=busan_geojson,

        locations="지역",
        featureidkey="properties.지역",

        color=metric,

        animation_frame="연도표시",

        hover_name="지역",

        hover_data={
            "사업자수": ":,.0f",
            "전월성장률": ":.2f",
            "전년동월성장률": ":.2f",
            "누적성장률": ":.2f",
            "연도표시": False
        },

        color_continuous_scale="RdYlGn",

        range_color=[
            -limit,
            limit
        ],

        center={
            "lat": 35.18,
            "lon": 129.07
        },

        zoom=8.6,

        map_style="carto-positron",

        opacity=0.78,

        labels={
            "전년동월성장률": "YoY 성장률(%)",
            "전월성장률": "MoM 성장률(%)",
            "누적성장률": "누적 성장률(%)"
        },

        title=f"부산 지역별 {industry} 사업자 성장 추이"
    )


    fig.update_layout(
        height=700,
        margin=dict(
            l=0,
            r=0,
            t=60,
            b=0
        )
    )

    fig.show()

In [14]:
# ============================================================
# 12. 월별 애니메이션 지도
# ============================================================

def monthly_growth_map(
    industry="전체",
    metric="전년동월성장률"
):

    data = viz[
        viz["업종"] == industry
    ].copy()

    data = data.sort_values("기준일")

    limit = np.nanmax(
        np.abs(data[metric])
    )

    if pd.isna(limit) or limit == 0:
        limit = 1

    fig = px.choropleth_map(
        data,
        geojson=busan_geojson,
        locations="지역",
        featureidkey="properties.지역",
        color=metric,
        animation_frame="연월표시",
        hover_name="지역",

        hover_data={
            "사업자수": ":,.0f",
            "전월성장률": ":.2f",
            "전년동월성장률": ":.2f",
            "누적성장률": ":.2f",
            "연월표시": False
        },

        color_continuous_scale="RdYlGn",
        range_color=[-limit, limit],

        center={
            "lat": 35.18,
            "lon": 129.07
        },

        zoom=8.6,
        map_style="carto-positron",
        opacity=0.78,

        title=f"부산 {industry} 월별 성장 변화",

        labels={
            metric: "성장률(%)"
        }
    )

    fig.update_layout(
        height=750,
        margin=dict(
            l=0,
            r=0,
            t=60,
            b=0
        )
    )

    return fig

In [15]:
import ipywidgets as widgets
from IPython.display import display, clear_output

industry_widget = widgets.Dropdown(
    options=[
        "전체",
        "쇼핑업",
        "서비스업",
        "식음료업",
        "의료웰니스업",
        "운송업",
        "여행/숙박업"
    ],
    value="전체",
    description="업종:"
)

metric_widget = widgets.Dropdown(
    options={
        "전년동월 성장률": "전년동월성장률",
        "전월 성장률": "전월성장률",
        "2022-01 대비 누적 성장률": "누적성장률"
    },
    value="전년동월성장률",
    description="지표:"
)

button = widgets.Button(
    description="지도 생성",
    button_style="primary"
)

output = widgets.Output()

def draw_map(b):
    with output:
        clear_output(wait=True)

        print("지도를 생성하고 있습니다...")

        fig = monthly_growth_map(
            industry=industry_widget.value,
            metric=metric_widget.value
        )

        clear_output(wait=True)
        fig.show()

button.on_click(draw_map)

display(
    industry_widget,
    metric_widget,
    button,
    output
)

Dropdown(description='업종:', options=('전체', '쇼핑업', '서비스업', '식음료업', '의료웰니스업', '운송업', '여행/숙박업'), value='전체')

Dropdown(description='지표:', options={'전년동월 성장률': '전년동월성장률', '전월 성장률': '전월성장률', '2022-01 대비 누적 성장률': '누적성장률'}, …

Button(button_style='primary', description='지도 생성', style=ButtonStyle())

Output()

In [16]:
# ============================================================
# 13. 지역별 성장지수
# ============================================================

def region_growth_chart(
    region="해운대구"
):

    data = viz[
        viz["지역"] == region
    ].copy()


    fig = px.line(

        data,

        x="기준일",

        y="성장지수",

        color="업종",

        hover_name="업종",

        hover_data={
            "사업자수": ":,.0f",
            "전년동월성장률": ":.2f",
            "누적성장률": ":.2f"
        },

        title=f"{region} 업종별 사업자 성장지수",

        labels={
            "기준일": "",
            "성장지수": "성장지수 (2022-01 = 100)",
            "업종": "업종"
        }
    )


    fig.add_hline(
        y=100,

        line_dash="dash",

        annotation_text="2022-01 기준"
    )


    fig.update_layout(
        height=550,

        hovermode="x unified"
    )


    fig.show()

In [17]:
region_widget = widgets.Dropdown(

    options=BUSAN_REGIONS,

    value="해운대구",

    description="지역:"
)


widgets.interact(
    region_growth_chart,
    region=region_widget
)

interactive(children=(Dropdown(description='지역:', index=8, options=('중구', '서구', '동구', '영도구', '부산진구', '동래구', '남…

<function __main__.region_growth_chart(region='해운대구')>

In [18]:
# ============================================================
# 14. 지역 × 연도 성장률 Heatmap
# ============================================================

heat_data = (
    annual[
        annual["업종"] == "전체"
    ]
    .pivot(
        index="지역",
        columns="연도",
        values="전년동월성장률"
    )
)


heat_data = heat_data.reindex(
    BUSAN_REGIONS
)


limit = np.nanmax(
    np.abs(heat_data.values)
)


fig = px.imshow(

    heat_data,

    text_auto=".1f",

    aspect="auto",

    color_continuous_scale="RdYlGn",

    zmin=-limit,
    zmax=limit,

    labels={
        "x": "연도",
        "y": "지역",
        "color": "YoY 성장률(%)"
    },

    title="부산 지역별 사업자 성장률 Heatmap"
)


fig.update_layout(
    height=650
)


fig.show()

In [19]:
# fig = monthly_growth_map(
#     industry="전체",
#     metric="전년동월성장률"
# )

# fig.show()

In [20]:
fig.write_html(
    "/content/부산_지역별_사업자성장지도.html"
)

In [21]:
from google.colab import files

files.download(
    "/content/부산_지역별_사업자성장지도.html"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
print("지도 생성 시작")

map_fig = monthly_growth_map(
    industry="전체",
    metric="전년동월성장률"
)

print("지도 객체 생성 완료")

map_fig.write_html(
    "/content/부산_사업자성장_월별지도.html"
)

print("HTML 저장 완료")

지도 생성 시작
지도 객체 생성 완료
HTML 저장 완료
